# 04. RQ4 Modeling: Is Divergence Structured?

The modeling section is not a production recommender. It is a diagnostic
test: if high-divergence users can be predicted from observable profile,
exposure, behavior, and weighted profile-state features, then the PEC gap
is structured rather than pure noise.

The emphasis is interpretation and validation, not model complexity. Treat every model output as a bounded audit signal.


<!-- PEC_RERUN_SAFETY_V1 -->

## Execution safety note

This notebook is an authorized private rerun notebook. It can use real processed data when `PEC_GAP_PROCESSED_DIR` or `PEC_GAP_PRIVATE_ROOT` is set, but notebook-generated outputs should stay under `tmp/repro_deep/...` unless you intentionally publish approved aggregate artifacts. The manuscript-facing `reports/tables/`, `reports/figures/`, and Overleaf folders are treated as read-only reference artifacts in this notebook.

**Public-package behavior:** If authorized processed data are unavailable, this notebook skips private-data computation cells without raising an error. This silent skip is expected and does not reproduce manuscript estimates. For a fully runnable bundled-data example, open `../01_metric_walkthrough_synthetic.ipynb` or run `python scripts/run_public_synthetic_pipeline.py` from the repository root.


<!-- PEC_NOTEBOOK_EXPLANATION_V1:modeling_guide -->

## Reader guide: what the model is and is not

The RQ4 model is a **diagnostic audit model**, not a production recommender. Its purpose is to test whether high preference-consumption divergence is structured by observable profile, exposure, behavior, and weighted profile-state features.

The model should be read as evidence for this question:

> Are high-divergence users random noise, or do observable traces partly identify them?

It should not be read as a targeting model, a causal model, or a new ranking architecture.

<!-- PEC_NOTEBOOK_EXPLANATION_V1:model_validation_terms -->

## Model validation terms

- **AUC:** probability that the model ranks a randomly chosen high-divergence user above a randomly chosen lower-divergence user. AUC is threshold-independent.
- **AUPRC:** area under the precision-recall curve. This is useful when positive labels are relatively rare or class balance matters.
- **Cross-validation:** repeated train/test splits used to estimate stability.
- **Temporal split:** trains on earlier periods and tests on later periods to reduce same-window overfitting concerns.
- **Leakage features:** variables too close to the target definition. These are excluded or separated so the model does not trivially reconstruct the label.
- **Feature importance / SHAP-style summary:** used for interpretation of which feature groups carry signal, not for causal explanation.

The strongest same-window AUC is intentionally interpreted with caution because weighted profile state is close to the preference-consumption construct being audited.

In [ ]:
from pathlib import Path
import os
import ast
import json
import math
import sys
import warnings
from collections import Counter


def find_repo_root() -> Path:
    """Find the public/release repository root, independent of private data location."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "reports").exists() and (candidate / "notebooks").exists() and (candidate / "scripts").exists():
            return candidate
    env_repo = os.environ.get("PEC_GAP_REPO_ROOT")
    if env_repo and Path(env_repo).exists():
        return Path(env_repo)
    raise FileNotFoundError("Could not find the PEC repository root. Set PEC_GAP_REPO_ROOT if needed.")


def find_processed_dir(repo_root: Path) -> Path:
    """Find authorized private processed tables without requiring them to live in the repo."""
    exact = os.environ.get("PEC_GAP_PROCESSED_DIR")
    if exact:
        return Path(exact)
    private_root = os.environ.get("PEC_GAP_PRIVATE_ROOT")
    if private_root:
        return Path(private_root) / "data" / "processed"
    local_config = repo_root / ".pecgap_local_paths.json"
    if local_config.exists():
        try:
            cfg = json.loads(local_config.read_text(encoding="utf-8"))
            if cfg.get("processed_dir"):
                return Path(cfg["processed_dir"])
            if cfg.get("private_root"):
                return Path(cfg["private_root"]) / "data" / "processed"
        except Exception as exc:
            print(f"Warning: could not read {local_config}: {exc}")
    return repo_root / "data" / "processed"


ROOT = find_repo_root()
PROCESSED = find_processed_dir(ROOT)
PKG = ROOT / ".python_packages"
if PKG.exists() and sys.version_info[:2] == (3, 12) and str(PKG) not in sys.path:
    # The project package cache is compiled for Python 3.12.
    # Anaconda/Jupyter kernels on this machine are often Python 3.10, so
    # we only add the cache when the ABI matches.
    sys.path.insert(0, str(PKG))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Read-only manuscript/reference artifacts. These are used for comparison.
# For authorized reruns, prefer the private project's reports/tables when the
# processed data live outside this public/release repo. This keeps row-level
# validation tables out of GitHub while still letting the notebooks compare
# regenerated values against the private run artifacts.
PRIVATE_ROOT = PROCESSED.parent.parent if PROCESSED.name == "processed" and PROCESSED.parent.name == "data" else ROOT
PRIVATE_TABLES = PRIVATE_ROOT / "reports" / "tables"
PRIVATE_FIGURES = PRIVATE_ROOT / "reports" / "figures"
PUBLIC_TABLES = ROOT / "reports" / "tables"
PUBLIC_FIGURES = ROOT / "reports" / "figures"
TABLES = PRIVATE_TABLES if PRIVATE_TABLES.exists() else PUBLIC_TABLES
FIGURES = PRIVATE_FIGURES if PRIVATE_FIGURES.exists() else PUBLIC_FIGURES

# Notebook-generated outputs go here by default, not into reports/tables or Overleaf.
NOTEBOOK_STEM = "04_modeling_rq4_from_features"
SCRATCH = ROOT / "tmp" / "repro_deep" / NOTEBOOK_STEM
SCRATCH_TABLES = SCRATCH / "tables"
SCRATCH_FIGURES = SCRATCH / "figures"
SCRATCH_TABLES.mkdir(parents=True, exist_ok=True)
SCRATCH_FIGURES.mkdir(parents=True, exist_ok=True)

PRIVATE_DATA_AVAILABLE = PROCESSED.exists()

print(f"Repository root: {ROOT}")
print(f"Processed data: {PROCESSED}")
print(f"Processed data exists: {PRIVATE_DATA_AVAILABLE}")
print(f"Read-only reference tables: {TABLES}")
print(f"Scratch output tables: {SCRATCH_TABLES}")
print(f"Scratch output figures: {SCRATCH_FIGURES}")
if not PRIVATE_DATA_AVAILABLE:
    display(Markdown(
        "**Private processed data not found.** This notebook is real rerun code, "
        "but it needs authorized processed CSVs. Set `PEC_GAP_PROCESSED_DIR` to the exact "
        "processed-data folder, or set `PEC_GAP_PRIVATE_ROOT` to a root containing `data/processed/`. Remaining private-data computation cells will be skipped; use the public synthetic pipeline for a fully runnable bundled-data example."
    ))


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    def safe_parse(value):
        '''Parse JSON-like strings stored in CSV cells.''';
        if value is None or (isinstance(value, float) and math.isnan(value)):
            return None
        text = str(value).strip()
        if not text:
            return None
        for parser in (json.loads, ast.literal_eval):
            try:
                return parser(text)
            except Exception:
                continue
        return None
    
    
    def truthy(series: pd.Series) -> pd.Series:
        '''Normalize common true-like CSV values to booleans.''';
        return series.astype(str).str.lower().isin(["true", "1", "yes", "y"])
    
    
    def pref_set(row: pd.Series, primary: str, fallback: str) -> set[str]:
        '''Return a stated preference set from explicit selections or weighted profile state.
    
        primary is usually the user-selection column. fallback is the current weighted
        profile-state column. For a weighted dictionary, small positive weights are
        treated as selected dimensions for set-overlap analyses.
        '''
        parsed = safe_parse(row.get(primary, ""))
        if parsed is None:
            parsed = safe_parse(row.get(fallback, ""))
        if isinstance(parsed, dict):
            out = set()
            for key, value in parsed.items():
                try:
                    if float(value) > 0.01:
                        out.add(str(key))
                except Exception:
                    out.add(str(key))
            return out or set(map(str, parsed.keys()))
        if isinstance(parsed, list):
            out = set()
            for item in parsed:
                if isinstance(item, dict) and "value" in item:
                    out.add(str(item["value"]))
                else:
                    out.add(str(item))
            return out
        return set()
    
    
    def normalize_dist(values: dict[str, float]) -> dict[str, float]:
        clean = {}
        for key, value in values.items():
            try:
                val = float(value)
            except Exception:
                continue
            if key and val > 0:
                clean[str(key)] = val
        total = sum(clean.values())
        if total <= 0:
            return {}
        return {key: val / total for key, val in clean.items()}
    
    
    def parse_weight_dist(value) -> dict[str, float]:
        '''Parse a weighted preference dictionary such as {'Politics': 0.7, ...}.''';
        parsed = safe_parse(value)
        if isinstance(parsed, dict):
            return normalize_dist({str(k): v for k, v in parsed.items()})
        if isinstance(parsed, list):
            vals = []
            for item in parsed:
                if isinstance(item, dict) and "value" in item:
                    vals.append(str(item["value"]))
                else:
                    vals.append(str(item))
            if vals:
                return {key: 1 / len(vals) for key in vals if key}
        return {}
    
    
    def norm_entropy(values) -> float:
        '''Normalized entropy in [0, 1]. Higher means more diverse.''';
        values = pd.Series(values).dropna().astype(str)
        values = values[values != ""]
        if values.empty:
            return np.nan
        counts = values.value_counts().to_numpy(dtype=float)
        probs = counts / counts.sum()
        ent = -(probs * np.log(probs)).sum()
        max_ent = np.log(len(counts)) if len(counts) > 1 else 1.0
        return float(ent / max_ent) if max_ent else 0.0
    
    
    def norm_entropy_dist(dist: dict[str, float]) -> float:
        values = np.array([v for v in dist.values() if v > 0], dtype=float)
        if len(values) == 0:
            return 0.0
        values = values / values.sum()
        ent = float(-(values * np.log(values)).sum())
        denom = math.log(len(values)) if len(values) > 1 else 1.0
        return ent / denom if denom else 0.0
    
    
    def top_share(values) -> float:
        '''Share of the most common item. Higher means more concentrated.''';
        values = pd.Series(values).dropna().astype(str)
        values = values[values != ""]
        if values.empty:
            return np.nan
        return float(values.value_counts().iloc[0] / len(values))
    
    
    def hhi(values) -> float:
        '''Herfindahl-Hirschman concentration index. Higher means more concentrated.''';
        values = pd.Series(values).dropna().astype(str)
        values = values[values != ""]
        if values.empty:
            return np.nan
        probs = values.value_counts(normalize=True).to_numpy(dtype=float)
        return float(np.sum(probs * probs))
    
    
    def jaccard(a: set[str], b: set[str]) -> float:
        if not a or not b:
            return np.nan
        return len(a & b) / len(a | b)
    
    
    def distribution_from_values(values) -> dict[str, float]:
        series = pd.Series(values).dropna().astype(str)
        series = series[series != ""]
        if series.empty:
            return {}
        counts = series.value_counts()
        return {str(k): float(v) / float(counts.sum()) for k, v in counts.items()}
    
    
    def cosine(a: dict[str, float], b: dict[str, float]) -> float:
        keys = sorted(set(a) | set(b))
        if not keys:
            return np.nan
        av = np.array([a.get(k, 0.0) for k in keys], dtype=float)
        bv = np.array([b.get(k, 0.0) for k in keys], dtype=float)
        denom = np.linalg.norm(av) * np.linalg.norm(bv)
        return float(np.dot(av, bv) / denom) if denom > 0 else np.nan
    
    
    def js_divergence(a: dict[str, float], b: dict[str, float]) -> float:
        '''Jensen-Shannon divergence between two discrete distributions.''';
        keys = sorted(set(a) | set(b))
        if not keys:
            return np.nan
        p = np.array([a.get(k, 0.0) for k in keys], dtype=float)
        q = np.array([b.get(k, 0.0) for k in keys], dtype=float)
        if p.sum() <= 0 or q.sum() <= 0:
            return np.nan
        p = p / p.sum()
        q = q / q.sum()
        m = 0.5 * (p + q)
        def kl(x, y):
            mask = x > 0
            return float((x[mask] * np.log2(x[mask] / y[mask])).sum())
        return 0.5 * kl(p, m) + 0.5 * kl(q, m)
    
    
    def dist_top_key(dist: dict[str, float]) -> str:
        return max(dist.items(), key=lambda item: item[1])[0] if dist else ""
    
    
    def classify_surface(row: pd.Series) -> str:
        '''Map raw app parameters to the surface categories used in the paper.''';
        source = str(row.get("p_source", "") or "").lower()
        section = str(row.get("section", "") or "").lower()
        message = str(row.get("message_type", "") or "").lower()
        is_headline = str(row.get("is_headline", "") or "").lower() in {"1.0", "1", "true", "yes"}
        if "push" in message or "notification" in source or "notification" in section:
            return "notification"
        if is_headline or "headline" in section:
            return "headline"
        if section.startswith("category_"):
            return "category"
        if "for_you" in section or source == "home":
            return "home/feed"
        if "related" in section or source == "news_detail":
            return "article-detail"
        if "publisher" in section or source in {"newsroom", "source"}:
            return "newsroom"
        if "podcast" in section or "podcast" in source:
            return "podcast"
        if "insight" in section or "insight" in source:
            return "insights"
        if "search" in section or source == "search":
            return "search"
        return "other/unknown"
    
    
    def mask_id(value: str) -> str:
        '''Mask user identifiers before showing examples in a notebook.''';
        text = str(value)
        if len(text) <= 8:
            return text[:2] + "***"
        return text[:4] + "***" + text[-4:]


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold, cross_validate
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # Load the engineered feature table created by the pipeline.
    # We also rebuild weighted preference features in this notebook so the
    # user can see how the extra model signal enters the feature matrix.
    features = pd.read_csv(TABLES / "rq4_divergence_model_features.csv")
    display(features.head())
    display(features[["divergence_score", "high_divergence", "click_count", "rec_count"]].describe().T)


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # Recompute weighted preference features, then merge.
    profiles = pd.read_csv(PROCESSED / "user_profiles_clean.csv", dtype=str).fillna("")
    clicks = pd.read_csv(PROCESSED / "click_events_enriched.csv", dtype=str).fillna("")
    matched_clicks = clicks.loc[truthy(clicks["news_matched"])].copy()
    profile_map = profiles.set_index("user_id")
    
    weighted_rows = []
    for user_id, group in matched_clicks.groupby("user_id"):
        if user_id not in profile_map.index:
            continue
        profile = profile_map.loc[user_id]
        cat_pref = parse_weight_dist(profile.get("interested_categories", ""))
        pub_pref = parse_weight_dist(profile.get("preferred_news_sources", ""))
        cat_click = distribution_from_values(group["category_final"])
        pub_click = distribution_from_values(group["publisher_id_final"])
        if not cat_click:
            continue
        top_click_cat = dist_top_key(cat_click)
        top_pref_cat = dist_top_key(cat_pref)
        top3_click_cats = [key for key, _ in Counter(group["category_final"].dropna().astype(str)).most_common(3)]
        weighted_rows.append({
            "user_id": user_id,
            "category_pref_entropy_weighted": norm_entropy_dist(cat_pref),
            "category_pref_max_weight": max(cat_pref.values()) if cat_pref else 0.0,
            "category_pref_salient_count": sum(v > 0.01 for v in cat_pref.values()),
            "publisher_pref_entropy_weighted": norm_entropy_dist(pub_pref),
            "publisher_pref_max_weight": max(pub_pref.values()) if pub_pref else 0.0,
            "publisher_pref_salient_count": sum(v > 0.001 for v in pub_pref.values()),
            "top_clicked_category_weight": cat_pref.get(top_click_cat, 0.0),
            "top_preference_category_click_share": cat_click.get(top_pref_cat, 0.0),
            "top3_clicked_category_pref_mass": sum(cat_pref.get(k, 0.0) for k in top3_click_cats),
            "category_pref_click_cosine": cosine(cat_pref, cat_click),
            "category_pref_click_js": js_divergence(cat_pref, cat_click),
            "publisher_pref_click_cosine": cosine(pub_pref, pub_click),
            "publisher_pref_click_js": js_divergence(pub_pref, pub_click),
        })
    
    weighted_features = pd.DataFrame(weighted_rows)
    model_df = features.merge(weighted_features, on="user_id", how="left")
    display(model_df.shape)


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # Feature families. Keeping these explicit makes it easy to see what each
    # ablation means.
    profile_features = [
        "num_stated_categories", "num_stated_publishers",
        "is_premium", "is_android", "is_korean_pref",
    ]
    exposure_features = [
        "same_day_available_rows", "same_day_hit_rate", "ever_recommended_hit_rate",
        "rec_count", "rec_category_entropy", "rec_publisher_entropy",
        "rec_top_category_share", "rec_top_publisher_share",
        "unique_rec_categories", "unique_rec_publishers",
    ]
    behavior_features = [
        "click_count", "click_category_entropy", "click_publisher_entropy",
        "click_top_category_share", "click_top_publisher_share",
        "avg_click_hour", "night_click_share", "active_click_days", "avg_daily_clicks",
        "headline_click_share", "breaking_click_share", "avg_click_position",
        "screen_entropy", "message_type_entropy",
    ]
    weighted_features_cols = [
        "category_pref_entropy_weighted", "category_pref_max_weight", "category_pref_salient_count",
        "publisher_pref_entropy_weighted", "publisher_pref_max_weight", "publisher_pref_salient_count",
        "top_clicked_category_weight", "top_preference_category_click_share",
        "top3_clicked_category_pref_mass", "category_pref_click_cosine", "category_pref_click_js",
        "publisher_pref_click_cosine", "publisher_pref_click_js",
    ]
    
    feature_sets = {
        "explicit_profile_only": profile_features,
        "weighted_profile_only": weighted_features_cols,
        "profile_plus_exposure": profile_features + exposure_features,
        "behavior_only": behavior_features,
        "full_without_weighted": profile_features + exposure_features + behavior_features,
        "full_plus_weighted": profile_features + exposure_features + behavior_features + weighted_features_cols,
    }
    feature_sets = {name: [c for c in cols if c in model_df.columns] for name, cols in feature_sets.items()}
    display(pd.DataFrame({"feature_set": feature_sets.keys(), "n_features": [len(v) for v in feature_sets.values()]}))


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    def eval_model(df, cols, y_col="high_divergence", model_name="random_forest", scale=False):
        x = df[cols]
        y = df[y_col].astype(int)
        minority = int(y.value_counts().min())
        if len(df) < 50 or y.nunique() < 2 or minority < 2:
            return {"n_users": len(df), "auc_mean": np.nan, "auprc_mean": np.nan, "f1_mean": np.nan, "cv_folds": 0}
    
        if model_name == "logistic":
            model = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)
            scale = True
        elif model_name == "gradient_boosting":
            model = GradientBoostingClassifier(random_state=42)
        elif model_name == "extra_trees":
            model = ExtraTreesClassifier(
                n_estimators=700,
                min_samples_leaf=3,
                max_features="sqrt",
                class_weight="balanced",
                bootstrap=False,
                random_state=92,
            )
        elif model_name == "random_forest":
            model = RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=4,
                max_features="sqrt",
                class_weight="balanced_subsample",
                random_state=42,
            )
        else:
            raise ValueError(f"Unknown model family: {model_name}")
    
        steps = [("imputer", SimpleImputer(strategy="median"))]
        if scale:
            steps.append(("scale", StandardScaler()))
        steps.append(("model", model))
        pipe = Pipeline(steps)
        n_splits = max(2, min(5, minority))
        scores = cross_validate(
            pipe,
            x,
            y,
            cv=StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42),
            scoring=["roc_auc", "average_precision", "f1"],
            n_jobs=1,
        )
        return {
            "n_users": len(df),
            "positive_rate": float(y.mean()),
            "auc_mean": float(scores["test_roc_auc"].mean()),
            "auc_std": float(scores["test_roc_auc"].std()),
            "auprc_mean": float(scores["test_average_precision"].mean()),
            "f1_mean": float(scores["test_f1"].mean()),
            "cv_folds": int(n_splits),
        }
    
    rows = []
    for set_name, cols in feature_sets.items():
        for model_name in ["logistic", "random_forest", "extra_trees", "gradient_boosting"]:
            result = eval_model(model_df, cols, model_name=model_name)
            result.update({"feature_set": set_name, "model": model_name})
            rows.append(result)
    cv_results = pd.DataFrame(rows).sort_values("auc_mean", ascending=False)
    display(cv_results)


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # Dense-history and click-threshold checks. These make the claim more bounded:
    # the model can be strong in same-window settings, but robustness checks
    # show how much signal remains under stricter user histories.
    threshold_rows = []
    for threshold in [1, 5, 10]:
        subset = model_df.loc[pd.to_numeric(model_df["click_count"], errors="coerce").fillna(0) >= threshold].copy()
        for model_name in ["logistic", "random_forest", "extra_trees", "gradient_boosting"]:
            result = eval_model(subset, feature_sets["full_plus_weighted"], model_name=model_name)
            result.update({"check": f"users_with_at_least_{threshold}_clicks", "model": model_name})
            threshold_rows.append(result)
    display(pd.DataFrame(threshold_rows))


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # Extreme mismatch diagnostic. This is an upper-bound task that compares
    # the clearest low- and high-divergence cases and drops middle cases.
    low = model_df["divergence_score"].quantile(0.25)
    high = model_df["divergence_score"].quantile(0.75)
    extreme = model_df.loc[(model_df["divergence_score"] <= low) | (model_df["divergence_score"] >= high)].copy()
    extreme["extreme_high_divergence"] = (extreme["divergence_score"] >= high).astype(int)
    
    extreme_rows = []
    for set_name in ["weighted_profile_only", "full_without_weighted", "full_plus_weighted"]:
        for model_name in ["logistic", "random_forest", "extra_trees", "gradient_boosting"]:
            result = eval_model(extreme, feature_sets[set_name], y_col="extreme_high_divergence", model_name=model_name)
            result.update({"feature_set": set_name, "label": "extreme_mismatch_upper_bound", "model": model_name})
            extreme_rows.append(result)
    display(pd.DataFrame(extreme_rows).sort_values("auc_mean", ascending=False))


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # Compare with saved paper/review tables.
    for filename in ["preference_weighted_model_cv.csv", "preference_weighted_stress_cv.csv", "optimization_model_grid.csv"]:
        path = TABLES / filename
        if path.exists():
            print(f"\n{filename}")
            display(pd.read_csv(path).head(10))


**How to read this model section**

- Higher AUC in the same-window setting shows that divergence is structured.
- Weighted profile-state features add signal, but they do not eliminate the gap.
- Dense-user and extreme-label checks are diagnostics, not production targeting claims.
- This is why the paper says the model is a bounded audit signal.


## Feature and model map: what RQ4 is testing

RQ4 starts from the user-level divergence label: users with high preference-consumption divergence versus lower-divergence users. The model then asks whether that label is predictable from observable traces.

The point is not to build a product model. The point is diagnostic:

> If observable profile, exposure, usage, and surface features can partly identify high-divergence users, then the PEC gap is structured rather than pure random noise.

This notebook therefore keeps feature families explicit and reports robustness checks. Some exploratory reruns can differ from the polished final manuscript tables; the final paper values are read from the locked handoff tables under `reports/tables` or the private run artifact folder.

In [ ]:
if PRIVATE_DATA_AVAILABLE:
    feature_family_map = pd.DataFrame([
        {
            "feature family": "profile_features",
            "examples": ", ".join(profile_features[:6]),
            "what it captures": "basic current profile/profile-support information",
            "interpretive caution": "not a versioned preference history",
        },
        {
            "feature family": "exposure_features",
            "examples": ", ".join(exposure_features[:6]),
            "what it captures": "logged recommendation availability and exposure distribution",
            "interpretive caution": "logged exposure is not total exposure",
        },
        {
            "feature family": "behavior_features",
            "examples": ", ".join(behavior_features[:6]),
            "what it captures": "click volume, timing, click distribution, and surface-use summaries",
            "interpretive caution": "behavioral summaries can be close to the label, so direct alignment quantities are excluded and sensitivities are checked",
        },
        {
            "feature family": "weighted_features",
            "examples": ", ".join(weighted_features[:6]),
            "what it captures": "weighted preference-click directional alignment summaries",
            "interpretive caution": "used for bounded diagnostics, not causal explanation",
        },
    ])
    display(feature_family_map)


## How to interpret AUC here

AUC is a ranking statistic: it asks whether the model tends to score a randomly chosen high-divergence user above a randomly chosen lower-divergence user. AUC 0.5 is chance-like; values above 0.5 indicate some structured signal.

For this paper, the important comparison is not “can we maximize AUC?” The important comparison is whether signal survives stricter checks:

- same-window model: strongest diagnostic setting;
- dense-history checks: require more click history;
- future-window check: asks whether earlier-window features carry signal for later-window labels;
- label-proximity checks: test whether empty profile states or support-count features trivially drive performance.

That is why the paper calls the model leakage-aware and bounded. It is evidence that divergence is patterned, not a production targeting system.

In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # AUTO-SAVE REPRO_DEEP SCRATCH OUTPUTS
    # This cell writes notebook-generated rerun outputs only to tmp/repro_deep/...
    # It does not write to reports/tables, reports/figures, or the Overleaf submission folder.
    _save_names = ['features', 'weighted_features', 'model_df', 'cv_results', 'threshold_rows', 'extreme_rows', 'feature_family_map']
    _saved_paths = []
    for _name in _save_names:
        if _name not in globals():
            continue
        _obj = globals()[_name]
        _path = SCRATCH_TABLES / f"{_name}.csv"
        try:
            if isinstance(_obj, pd.DataFrame):
                _obj.to_csv(_path, index=False)
            elif isinstance(_obj, pd.Series):
                _obj.to_frame().to_csv(_path)
            elif isinstance(_obj, list):
                pd.DataFrame(_obj).to_csv(_path, index=False)
            elif isinstance(_obj, dict):
                pd.DataFrame([_obj]).to_csv(_path, index=False)
            else:
                continue
            _saved_paths.append(_path)
        except Exception as exc:
            print(f"Could not save {_name}: {exc}")
    
    try:
        import matplotlib.pyplot as _plt
        for _idx, _num in enumerate(_plt.get_fignums(), 1):
            _fig = _plt.figure(_num)
            _fig_path = SCRATCH_FIGURES / f"{NOTEBOOK_STEM}_figure_{_idx:02d}.png"
            _fig.savefig(_fig_path, dpi=160, bbox_inches="tight")
            _saved_paths.append(_fig_path)
    except Exception as exc:
        print(f"Could not save open matplotlib figures: {exc}")
    
    print("Saved scratch artifacts:")
    for _path in _saved_paths:
        print(" -", _path)
